# Data Preparation: Preprocessing and Feature Engineering

## Data Preprocessing

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

In [4]:
DATA_PATH = Path("../data/AirQualityUCI.csv")

df = pd.read_csv(
    DATA_PATH,
    sep=";",
    decimal=",",
    na_values=-200
)

empty_columns = [
    column
    for column in ["Unnamed: 15", "Unnamed: 16"]
    if column in df.columns
]

df = (
    df
    .drop(columns=empty_columns)
    .dropna(how="all")
    .copy()
)

df["DateTime"] = pd.to_datetime(
    df["Date"].astype(str)
    + " "
    + df["Time"].astype(str).str.replace(".", ":", regex=False),
    format="%d/%m/%Y %H:%M:%S",
    errors="coerce"
)

df = (
    df
    .drop(columns=["Date", "Time"])
    .sort_values("DateTime")
    .reset_index(drop=True)
)

In [5]:
target_columns = [
    "CO(GT)",
    "C6H6(GT)",
    "NOx(GT)",
    "NO2(GT)",
]

sensor_features = [
    "PT08.S1(CO)",
    "PT08.S2(NMHC)",
    "PT08.S3(NOx)",
    "PT08.S4(NO2)",
    "PT08.S5(O3)",
]

environment_features = [
    "T",
    "RH",
    "AH",
]

ground_truth_columns = [
    "CO(GT)",
    "NMHC(GT)",
    "C6H6(GT)",
    "NOx(GT)",
    "NO2(GT)",
]

In [10]:
df.shape

(9357, 14)

In [7]:
df.head()

,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH,DateTime
0,2.6,1360.0,150.0,11.9,1046.0,166.0,1056.0,113.0,1692.0,1268.0,13.6,48.9,0.7578,2004-03-10 18:00:00
1,2.0,1292.0,112.0,9.4,955.0,103.0,1174.0,92.0,1559.0,972.0,13.3,47.7,0.7255,2004-03-10 19:00:00
2,2.2,1402.0,88.0,9.0,939.0,131.0,1140.0,114.0,1555.0,1074.0,11.9,54.0,0.7502,2004-03-10 20:00:00
3,2.2,1376.0,80.0,9.2,948.0,172.0,1092.0,122.0,1584.0,1203.0,11.0,60.0,0.7867,2004-03-10 21:00:00
4,1.6,1272.0,51.0,6.5,836.0,131.0,1205.0,116.0,1490.0,1110.0,11.2,59.6,0.7888,2004-03-10 22:00:00


In [8]:
df.dtypes

CO(GT)                  float64
PT08.S1(CO)             float64
NMHC(GT)                float64
C6H6(GT)                float64
PT08.S2(NMHC)           float64
NOx(GT)                 float64
PT08.S3(NOx)            float64
NO2(GT)                 float64
PT08.S4(NO2)            float64
PT08.S5(O3)             float64
T                       float64
RH                      float64
AH                      float64
DateTime         datetime64[us]
dtype: object

In [11]:
print("Invalid DateTime:", df["DateTime"].isna().sum())
print("Duplicate DateTime:", df["DateTime"].duplicated().sum())

Invalid DateTime: 0
Duplicate DateTime: 0


In [12]:
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Start:", df["DateTime"].min())
print("End:", df["DateTime"].max())
print("Sorted:", df["DateTime"].is_monotonic_increasing)
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate timestamps:", df["DateTime"].duplicated().sum())

time_gap = df["DateTime"].diff().value_counts().head()
time_gap

Rows: 9357
Columns: 14
Start: 2004-03-10 18:00:00
End: 2005-04-04 14:00:00
Sorted: True
Duplicate rows: 0
Duplicate timestamps: 0


DateTime
0 days 01:00:00    9356
Name: count, dtype: int64

## Feature Engineering

In [13]:
df["hour"] = df["DateTime"].dt.hour
df["day_of_week"] = df["DateTime"].dt.dayofweek
df["month"] = df["DateTime"].dt.month
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

In [14]:
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

df["month_sin"] = np.sin(2 * np.pi * (df["month"] - 1) / 12)
df["month_cos"] = np.cos(2 * np.pi * (df["month"] - 1) / 12)

In [15]:
time_feature_columns = [
    "hour",
    "day_of_week",
    "month",
    "is_weekend",
    "hour_sin",
    "hour_cos",
    "month_sin",
    "month_cos",
]

df[["DateTime"] + time_feature_columns].head()

,DateTime,hour,day_of_week,month,is_weekend,hour_sin,hour_cos,month_sin,month_cos
0,2004-03-10 18:00:00,18,2,3,0,-1.000000,-1.836970e-16,0.866025,0.5
1,2004-03-10 19:00:00,19,2,3,0,-0.965926,2.588190e-01,0.866025,0.5
2,2004-03-10 20:00:00,20,2,3,0,-0.866025,5.000000e-01,0.866025,0.5
3,2004-03-10 21:00:00,21,2,3,0,-0.707107,7.071068e-01,0.866025,0.5
4,2004-03-10 22:00:00,22,2,3,0,-0.500000,8.660254e-01,0.866025,0.5


In [18]:
feature_columns = (
    sensor_features
    + environment_features
    + ["hour_sin", "hour_cos", "month_sin", "month_cos", "is_weekend"]
)

feature_columns

['PT08.S1(CO)',
 'PT08.S2(NMHC)',
 'PT08.S3(NOx)',
 'PT08.S4(NO2)',
 'PT08.S5(O3)',
 'T',
 'RH',
 'AH',
 'hour_sin',
 'hour_cos',
 'month_sin',
 'month_cos',
 'is_weekend']

## Time Based Split
The data is divided based on chronological order:
- train: the earliest period.
- validation: the period following the training set.
- test: the final period.

In [20]:
train_ratio = 0.70
validation_ratio = 0.15
test_ratio = 0.15

In [21]:
n_rows = len(df)

train_end = int(n_rows * train_ratio)
validation_end = int(n_rows * (train_ratio + validation_ratio))

train_df = df.iloc[:train_end].copy()
validation_df = df.iloc[train_end:validation_end].copy()
test_df = df.iloc[validation_end:].copy()

In [26]:
split_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "rows": [len(train_df), len(validation_df), len(test_df)],
    "start": [train_df["DateTime"].min(), validation_df["DateTime"].min(), test_df["DateTime"].min()],
    "end": [train_df["DateTime"].max(), validation_df["DateTime"].max(), test_df["DateTime"].max()],
})

split_summary

,split,rows,start,end
0,train,6549,2004-03-10 18:00:00,2004-12-08 14:00:00
1,validation,1404,2004-12-08 15:00:00,2005-02-05 02:00:00
2,test,1404,2005-02-05 03:00:00,2005-04-04 14:00:00


## Causal Feature Imputation

In [28]:
missing_features_before = df[feature_columns].isna().sum()
missing_features_before.sort_values(ascending=False)

PT08.S1(CO)      366
PT08.S2(NMHC)    366
PT08.S3(NOx)     366
PT08.S4(NO2)     366
PT08.S5(O3)      366
T                366
RH               366
AH               366
hour_sin           0
hour_cos           0
month_sin          0
month_cos          0
is_weekend         0
dtype: int64

In [29]:
df_features = df[["DateTime"] + feature_columns].copy()

df_features[feature_columns] = (df_features[feature_columns].ffill())

In [31]:
remaining_missing = df_features[feature_columns].isna().sum()
remaining_missing.sort_values(ascending=False)

PT08.S1(CO)      0
PT08.S2(NMHC)    0
PT08.S3(NOx)     0
PT08.S4(NO2)     0
PT08.S5(O3)      0
T                0
RH               0
AH               0
hour_sin         0
hour_cos         0
month_sin        0
month_cos        0
is_weekend       0
dtype: int64

## Median Imputer and Scaling

In [33]:
train_features = df_features.iloc[:train_end].copy()
validation_features = df_features.iloc[train_end:validation_end].copy()
test_features = df_features.iloc[validation_end:].copy()

X_train_raw = train_features[feature_columns]
X_validation_raw = validation_features[feature_columns]
X_test_raw = test_features[feature_columns]

In [41]:
median_imputer = SimpleImputer(strategy="median")

X_train_imputed = median_imputer.fit_transform(X_train_raw)
X_validation_imputed = median_imputer.transform(X_validation_raw)
X_test_imputed = median_imputer.transform(X_test_raw)

In [42]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_imputed)
X_validation_scaled = scaler.transform(X_validation_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

In [45]:
X_train_processed = pd.DataFrame(
    X_train_scaled,
    columns=feature_columns,
    index=train_features.index,
)

X_validation_processed = pd.DataFrame(
    X_validation_scaled,
    columns=feature_columns,
    index=validation_features.index,
)

X_test_processed = pd.DataFrame(
    X_test_scaled,
    columns=feature_columns,
    index=test_features.index,
)

In [47]:
X_train_processed.head()

,PT08.S1(CO),PT08.S2(NMHC),PT08.S3(NOx),PT08.S4(NO2),PT08.S5(O3),T,RH,AH,hour_sin,hour_cos,month_sin,month_cos,is_weekend
0,1.199164,0.301788,0.797555,0.382354,0.696000,-1.060380,0.073403,-1.181534,-1.414910,-0.000317,1.181839,1.427979,-0.632658
1,0.888275,-0.039209,1.267049,-0.077010,-0.090886,-1.099326,0.004621,-1.273081,-1.366716,0.365671,1.181839,1.427979,-0.632658
2,1.391183,-0.099165,1.131771,-0.090825,0.180271,-1.281074,0.365725,-1.203075,-1.225421,0.706717,1.181839,1.427979,-0.632658
3,1.272314,-0.065440,0.940791,0.009337,0.523205,-1.397912,0.709634,-1.099624,-1.000653,0.999579,1.181839,1.427979,-0.632658
4,0.796837,-0.485128,1.390390,-0.315326,0.275973,-1.371948,0.686707,-1.093672,-0.707729,1.224301,1.181839,1.427979,-0.632658


In [48]:
print("Train missing:", X_train_processed.isna().sum().sum())
print("Validation missing:", X_validation_processed.isna().sum().sum())
print("Test missing:", X_test_processed.isna().sum().sum())

print("Train shape:", X_train_processed.shape)
print("Validation shape:", X_validation_processed.shape)
print("Test shape:", X_test_processed.shape)

Train missing: 0
Validation missing: 0
Test missing: 0
Train shape: (6549, 13)
Validation shape: (1404, 13)
Test shape: (1404, 13)


## Target-Specific Dataset Preparation

In [49]:
target_counts = []

for target in target_columns:
    target_counts.append({
        "target": target,
        "train_available": train_df[target].notna().sum(),
        "train_missing": train_df[target].isna().sum(),
        "validation_available": validation_df[target].notna().sum(),
        "validation_missing": validation_df[target].isna().sum(),
        "test_available": test_df[target].notna().sum(),
        "test_missing": test_df[target].isna().sum(),
    })

target_counts = pd.DataFrame(target_counts)
target_counts

,target,train_available,train_missing,validation_available,validation_missing,test_available,test_missing
0,CO(GT),5066,1483,1234,170,1374,30
1,C6H6(GT),6401,148,1263,141,1327,77
2,NOx(GT),5099,1450,1252,152,1367,37
3,NO2(GT),5096,1453,1252,152,1367,37


In [51]:
target_datasets = {}

for target in target_columns:
    train_mask = train_df[target].notna()
    validation_mask = validation_df[target].notna()
    test_mask = test_df[target].notna()

    target_datasets[target] = {
        "X_train": X_train_processed.loc[train_mask],
        "y_train": train_df.loc[train_mask, target].copy(),
        "X_validation": X_validation_processed.loc[validation_mask],
        "y_validation": validation_df.loc[validation_mask, target].copy(),
        "X_test": X_test_processed.loc[test_mask],
        "y_test": test_df.loc[test_mask, target].copy(),
    }

In [62]:
for target, data in target_datasets.items():
    print(
        target,
        "train=", data["X_train"].shape,
        "validation=", data["X_validation"].shape,
        "test=", data["X_test"].shape,
    )

CO(GT) train= (5066, 13) validation= (1234, 13) test= (1374, 13)
C6H6(GT) train= (6401, 13) validation= (1263, 13) test= (1327, 13)
NOx(GT) train= (5099, 13) validation= (1252, 13) test= (1367, 13)
NO2(GT) train= (5096, 13) validation= (1252, 13) test= (1367, 13)
